In [ ]:
!pip install grad-cam

In [ ]:
import os, cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

In [ ]:
IMG_DIR1 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1/"
IMG_DIR2 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2/"
MASK_DIR = "/kaggle/input/datasets/tschandl/ham10000-lesion-segmentations/HAM10000_segmentations_lesion_tschandl/"
CSV_PATH = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv"

df = pd.read_csv(CSV_PATH)
df['path'] = df['image_id'].apply(lambda x: os.path.join(IMG_DIR1, x+'.jpg') 
                                  if os.path.exists(os.path.join(IMG_DIR1, x+'.jpg')) 
                                  else os.path.join(IMG_DIR2, x+'.jpg'))
df['mask_path'] = df['image_id'].apply(lambda x: os.path.join(MASK_DIR, x+'_segmentation.png'))

In [ ]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['dx'])
NUM_CLASSES = df['label'].nunique()

In [ ]:
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_aug = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=40, p=0.7),
    A.RandomBrightnessContrast(p=0.5),
    A.CoarseDropout(p=0.5),
    A.Normalize(),
    ToTensorV2()
])

valid_aug = A.Compose([
    A.Resize(256, 256),
    A.Normalize(),
    ToTensorV2()
])

In [ ]:
import torch
from torch.utils.data import Dataset
import cv2
import numpy as np

class SkinDataset(Dataset):
    def __init__(self, df, aug=None, is_segmentation=True):
        """
        df: dataframe with columns 'path', 'mask_path', 'label'
        aug: albumentations augmentation
        is_segmentation: whether to return mask for segmentation
        """
        self.df = df
        self.aug = aug
        self.is_segmentation = is_segmentation

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
    
        # Load image
        img = cv2.imread(row['path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
        # Load mask
        if self.is_segmentation:
            mask = cv2.imread(row['mask_path'], cv2.IMREAD_GRAYSCALE)
            if mask is None:
                mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
            mask = mask.astype(np.float32)/255.0
            mask = np.expand_dims(mask, axis=-1)
        else:
            mask = np.zeros((img.shape[0], img.shape[1], 1), dtype=np.float32)
    
        # Apply augmentation
        if self.aug:
            augmented = self.aug(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
    
        # Convert to tensor safely
        mask = torch.as_tensor(mask, dtype=torch.float).permute(2,0,1)  # (1,H,W)
    
        # Label
        label = torch.tensor(row['label'], dtype=torch.long)
    
        return img, mask, label

In [ ]:
BATCH_SIZE = 4  # safe for Kaggle GPU

train_dataset = SkinDataset(train_df, train_aug, is_segmentation=True)
val_dataset   = SkinDataset(val_df, valid_aug, is_segmentation=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
#U-NET++ (BETTER SEGMENTATION)
class UNetPP(nn.Module):
    def __init__(self, encoder_name='resnet34'):
        super().__init__()
        self.encoder = timm.create_model(encoder_name, pretrained=True, features_only=True)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512,256,2,2), nn.ReLU(),
            nn.ConvTranspose2d(256,128,2,2), nn.ReLU(),
            nn.ConvTranspose2d(128,64,2,2), nn.ReLU(),
            nn.ConvTranspose2d(64,32,2,2), nn.ReLU(),
            nn.ConvTranspose2d(32,1,1)
        )
    def forward(self,x):
        feat = self.encoder(x)[-1]
        return self.decoder(feat)

In [ ]:
# LOSS
bce = nn.BCEWithLogitsLoss()
def dice_loss(pred,target):
    pred = torch.sigmoid(pred)
    smooth = 1.
    inter = (pred*target).sum()
    return 1-(2*inter+smooth)/(pred.sum()+target.sum()+smooth)
def seg_loss(pred,target):
    return bce(pred,target)+dice_loss(pred,target)

In [ ]:
device = 'cuda'

In [ ]:
# SEGMENTATION MODEL
seg_model = UNetPP().to(device)
opt_seg = torch.optim.Adam(seg_model.parameters(), lr=1e-4)

In [ ]:
import torch.nn.functional as F  

# 11️⃣ SEGMENTATION TRAINING
EPOCHS_SEG = 10
best_seg_loss = np.inf

for epoch in range(EPOCHS_SEG):
    seg_model.train()
    total = 0
    for img, mask, _ in train_loader:
        img, mask = img.to(device), mask.to(device)
        pred = seg_model(img)
        mask_resized = F.interpolate(mask, size=pred.shape[2:], mode='nearest')  # 🔹 fix size
        loss = seg_loss(pred, mask_resized)
        opt_seg.zero_grad()
        loss.backward()
        opt_seg.step()
        total += loss.item()
    avg_loss = total/len(train_loader)
    print(f"[SEG] Epoch {epoch} Loss: {avg_loss:.4f}")
    
    # save best segmentation model
    if avg_loss < best_seg_loss:
        torch.save(seg_model.state_dict(), 'best_seg_model.pth')
        best_seg_loss = avg_loss

In [ ]:
import torch.nn.functional as F

EPOCHS_SEG = 10
best_seg_loss = np.inf

for epoch in range(EPOCHS_SEG):
    # -------------------
    # 1️⃣ Training
    # -------------------
    seg_model.train()
    total_train_loss = 0
    for img, mask, _ in train_loader:
        img, mask = img.to(device), mask.to(device)
        pred = seg_model(img)
        mask_resized = F.interpolate(mask, size=pred.shape[2:], mode='nearest')
        loss = seg_loss(pred, mask_resized)
        
        opt_seg.zero_grad()
        loss.backward()
        opt_seg.step()
        
        total_train_loss += loss.item()
    
    avg_train_loss = total_train_loss / len(train_loader)
    
    # -------------------
    # 2️⃣ Validation
    # -------------------
    seg_model.eval()
    total_val_loss = 0
    total_dice = 0
    with torch.no_grad():
        for img, mask, _ in val_loader:
            img, mask = img.to(device), mask.to(device)
            pred = seg_model(img)
            mask_resized = F.interpolate(mask, size=pred.shape[2:], mode='nearest')
            loss = seg_loss(pred, mask_resized)
            total_val_loss += loss.item()
            
            # Dice score for validation
            pred_sigmoid = torch.sigmoid(pred)
            pred_bin = (pred_sigmoid > 0.5).float()
            inter = (pred_bin * mask_resized).sum(dim=(1,2,3))
            dice = (2*inter + 1e-6) / (pred_bin.sum(dim=(1,2,3)) + mask_resized.sum(dim=(1,2,3)) + 1e-6)
            total_dice += dice.mean().item()
    
    avg_val_loss = total_val_loss / len(val_loader)
    avg_val_dice = total_dice / len(val_loader)
    
    # -------------------
    # 3️⃣ Print metrics
    # -------------------
    print(f"[SEG] Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Dice: {avg_val_dice:.4f}")
    
    # -------------------
    # 4️⃣ Save best model
    # -------------------
    if avg_val_loss < best_seg_loss:
        torch.save(seg_model.state_dict(), 'best_seg_model.pth')
        best_seg_loss = avg_val_loss